In [1]:
# Cell 1 — Imports
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

df = pd.read_csv('../data/raw/creditcard.csv')
print("Loaded:", df.shape)

Loaded: (284807, 31)


In [2]:
# Cell 2 — Scale Amount and Time
scaler = StandardScaler()

df['Amount_scaled'] = scaler.fit_transform(df[['Amount']])
df['Time_scaled'] = scaler.fit_transform(df[['Time']])

# Drop originals
df.drop(columns=['Amount', 'Time'], inplace=True)

print("Scaled. New columns:", df.columns.tolist())

Scaled. New columns: ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Class', 'Amount_scaled', 'Time_scaled']


In [3]:
# Cell 3 — Check missing values
print(df.isnull().sum().sum(), "missing values")

0 missing values


In [4]:
# Cell 4 — Split features and target
X = df.drop(columns=['Class'])
y = df['Class']

print("X shape:", X.shape)
print("y distribution:\n", y.value_counts())

X shape: (284807, 30)
y distribution:
 Class
0    284315
1       492
Name: count, dtype: int64


In [5]:
# Cell 5 — Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (227845, 30)
Test size: (56962, 30)


In [6]:
# Cell 6 — Handle class imbalance with SMOTE
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

print("After SMOTE:")
print(pd.Series(y_train_res).value_counts())

After SMOTE:
Class
0    227451
1    227451
Name: count, dtype: int64


In [ ]:
# Cell 7 — Save processed data
import os

os.makedirs('../data/processed', exist_ok=True)

# Save test set as-is (no SMOTE on test data, ever)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

# Save resampled train set
X_train_res_df = pd.DataFrame(X_train_res, columns=X.columns)
y_train_res_df = pd.Series(y_train_res, name='Class')

X_train_res_df.to_csv('../data/processed/X_train.csv', index=False)
y_train_res_df.to_csv('../data/processed/y_train.csv', index=False)

print("✅ Saved to data/processed/")